# Lab | Langchain Evaluation

## Intro

Pick different sets of data and re-run this notebook. The point is for you to understand all steps involve and the many different ways one can and should evaluate LLM applications.

What did you learn? - Let's discuss that in class

## LangChain: Evaluation

### Outline:

* Example generation
* Manual evaluation (and debuging)
* LLM-assisted evaluation

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv
load_dotenv()
DATA_DIR = Path.cwd() / "data"
assert DATA_DIR.is_dir(), "Start Jupyter from the repository root."
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your environment or .env first."
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

### Setup and execution
Use Python 3.11, install `requirements.txt`, and select that environment as the notebook kernel. Set `OPENAI_API_KEY` in a local `.env` file. Run all cells from the repository root. The pinned versions preserve the lab's LangChain chain APIs; Ragas uses its documented dataset API.

Execution status: source data and Python syntax were checked locally. Live model outputs and grades have not been generated because the environment has no OpenAI key or LangChain/Ragas dependencies. No scores below are fabricated. Two generated catalog questions and small evaluation batches limit API cost.

### Example 1

#### Create our QandA application

In [ ]:
from langchain.chains import RetrievalQA
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import CSVLoader, TextLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.prompts import PromptTemplate
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
)
llm = ChatOpenAI(model=MODEL, temperature=0, max_tokens=256)

In [ ]:
loader = CSVLoader(file_path=str(DATA_DIR / "OutdoorClothingCatalog_1000.csv"))
data = loader.load()

In [ ]:
# Keep each catalog product intact so its name and attributes stay together.

In [ ]:
catalog_store = FAISS.from_documents(data, embeddings)

In [ ]:
qa_prompt = PromptTemplate.from_template(
    "Answer using only the context. If it does not contain the answer, say "
    "'I do not know based on the supplied context.' Keep the answer concise.\n"
    "Context: {context}\nQuestion: {question}\nAnswer:"
)
qa = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff",
    retriever=catalog_store.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True,
    chain_type_kwargs={"prompt": qa_prompt, "document_separator": "\n<<<<>>>>>\n"},
)

#### Coming up with test datapoints

In [ ]:
data[10]

In [ ]:
data[11]

#### Hard-coded examples

In [ ]:
from langchain.prompts import PromptTemplate

In [ ]:
hardcoded_examples = [
    {"query": "Does the Cozy Comfort Pullover Set, Stripe have side pockets?", "answer": "Yes, the pants have side pockets."},
    {"query": "What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?", "answer": "The DownTek collection."},
]
# Retrieval supplies evidence; few-shot answers alone cannot establish product facts.
query = "Is the Cozy Comfort Pullover Set available in different colors?"
color_result = qa.invoke({"query": query})
print(color_result["result"])

#### LLM-Generated examples

In [ ]:
from langchain.evaluation.qa import QAGenerateChain

In [ ]:
example_gen_chain = QAGenerateChain.from_llm(llm)

In [ ]:
# Generate only two questions to keep evaluation costs small.

In [ ]:
new_examples = [example_gen_chain.invoke({"doc": doc.page_content}) for doc in data[:2]]

In [ ]:
new_examples[0]

In [ ]:
data[0]

In [ ]:
d_flattened = [item["qa_pairs"] for item in new_examples]
assert all({"query", "answer"} <= set(item) for item in d_flattened)
d_flattened

#### Combine examples

In [ ]:
examples = hardcoded_examples + d_flattened
assert len({e["query"] for e in examples}) == len(examples)

In [ ]:
examples[0]

In [ ]:
first_prediction = qa.invoke({"query": examples[0]["query"]})
first_prediction["result"]

### Manual Evaluation - Fun part

In [ ]:
# Inspect retrieved evidence without printing verbose model traces.

In [ ]:
print("Question:", examples[0]["query"])
print("Reference:", examples[0]["answer"])
print("Prediction:", first_prediction["result"])
for doc in first_prediction["source_documents"]:
    print(doc.metadata, doc.page_content)
# Check whether the correct product was retrieved and whether every claim is supported.

Manual source review: catalog row 10 explicitly states that the pants have side pockets; row 11 names the DownTek collection. These references are supported by the supplied data. Color availability is not established by row 10, so a confident color claim would need additional retrieved evidence. Inspect the generated questions against their source products before trusting their reference answers.

### LLM assisted evaluation

In [ ]:
# Do not append generated examples again: duplicates would distort the score.

In [ ]:
examples

In [ ]:
predictions = [first_prediction] + qa.batch(
    [{"query": e["query"]} for e in examples[1:]], config={"max_concurrency": 2}
)

In [ ]:
predictions

In [ ]:
from langchain.evaluation.qa import QAEvalChain

In [ ]:
eval_chain = QAEvalChain.from_llm(llm)

In [ ]:
graded_outputs = eval_chain.evaluate(examples, predictions, question_key="query", answer_key="answer", prediction_key="result")

In [ ]:
graded_outputs

In [ ]:
for example, prediction, grade in zip(examples, predictions, graded_outputs):
    print("Question:", example["query"])
    print("Reference:", example["answer"])
    print("Prediction:", prediction["result"])
    print("Grade:", grade["results"], "\n")
grades = [g["results"].strip().upper() for g in graded_outputs]
print("Judge agreement rate:", sum(g == "CORRECT" for g in grades) / len(grades))
print("Unrecognized grades:", [g for g in grades if g not in {"CORRECT", "INCORRECT"}])

### Example 2
One can also easily evaluate your QA chains with the metrics offered in ragas

In [ ]:
nyc_docs = TextLoader(str(DATA_DIR / "nyc_text.txt"), encoding="utf-8").load()
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
nyc_chunks = splitter.split_documents(nyc_docs)
nyc_store = FAISS.from_documents(nyc_chunks, embeddings)
qa_chain = RetrievalQA.from_chain_type(
    llm=llm, retriever=nyc_store.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True, chain_type_kwargs={"prompt": qa_prompt},
)

In [ ]:
# testing it out

question = "How did New York City get its name?"
result = qa_chain.invoke({"query": question})
result["result"]

In [ ]:
result

Now in order to evaluate the qa system we generated a few relevant questions. We've generated a few question for you but feel free to add any you want.

In [ ]:
eval_questions = [
    "What is the population of New York City as of 2020?",
    "Which borough of New York City has the highest population?",
    "What is the economic significance of New York City?",
    "How did New York City get its name?",
    "What is the significance of the Statue of Liberty in New York City?",
]

eval_answers = [
    "8,804,190 in 2020.",
    "Brooklyn.",
    "New York City is a global financial center and a major center of commerce and international business.",
    "It was renamed New York in 1664 under British control, in honor of the Duke of York.",
    "It greeted millions of immigrants and symbolizes the United States and its ideals of liberty and peace.",
]
examples = [{"query": q, "ground_truth": a} for q, a in zip(eval_questions, eval_answers)]
assert len(eval_questions) == len(eval_answers)

In [ ]:
examples

#### Evaluate with Ragas datasets

Use the [Ragas 0.1.21 dataset API](https://docs.ragas.io/en/v0.1.21/getstarted/evaluation.html). Each row contains `question`, `answer`, `contexts` (strings), and `ground_truth` (one string). Context recall needs the reference answer. Faithfulness checks support in retrieved text, while answer relevancy checks whether the response addresses the question. Context precision checks whether relevant retrieved chunks rank ahead of irrelevant ones.

In [ ]:
result = qa_chain.invoke({"query": eval_questions[1]})
result["result"]

In [ ]:
def evaluation_row(example, prediction):
    return {
        "question": example["query"],
        "answer": prediction["result"],
        "contexts": [doc.page_content for doc in prediction["source_documents"]],
        "ground_truth": example["ground_truth"],
    }
result_updated = evaluation_row(examples[1], result)

In [ ]:
result_updated

In [ ]:
# Dependencies are pinned in requirements.txt.

In [ ]:
# Local Hugging Face embeddings are reused by Ragas.

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.run_config import RunConfig

def score_rows(rows, metrics):
    return evaluate(
        Dataset.from_list(rows), metrics=metrics, llm=llm, embeddings=embeddings,
        run_config=RunConfig(max_workers=2, timeout=120), raise_exceptions=True,
    )

1. Evaluate a single answer with its retrieved evidence.

In [ ]:
# Recheck the result that we are going to validate.
result

**Faithfulness**

In [ ]:
eval_result = score_rows([result_updated], [faithfulness])
eval_result

High faithfulness means the answer’s factual claims are supported by the retrieved context. It does not establish that the context is true or that the answer is complete.

In [ ]:
fake_result = {**result_updated, "answer": "Brooklyn has a population of 500 billion and is located on Mars."}
score_rows([fake_result], [faithfulness])

**Context recall**

In [ ]:
score_rows([result_updated], [context_recall])

Context recall measures how much of the reference answer the retrieved passages support. Replacing those passages with unrelated text should reduce recall; the actual score must be measured.

In [ ]:
fake_context = {**result_updated, "contexts": ["I love Christmas."]}
score_rows([fake_context], [context_recall])

2. Evaluate the complete NYC question set. Reuse predictions for every metric.

In [ ]:
predictions = qa_chain.batch([{"query": e["query"]} for e in examples], config={"max_concurrency": 2})
rows = [evaluation_row(e, p) for e, p in zip(examples, predictions)]
nyc_scores = score_rows(rows, [faithfulness, answer_relevancy, context_precision, context_recall])
nyc_scores

In [ ]:
report = nyc_scores.to_pandas()
report.to_csv("nyc_evaluation_results.csv", index=False)
report

### What I learned

The two datasets need different retrieval preparation: catalog rows are self-contained products, while the long NYC article needs overlapping chunks. A wrong answer can come from missing evidence or from ignoring correct evidence, so I should inspect retrieved passages before changing the prompt.

Manual review establishes trustworthy reference answers. Generated questions increase coverage cheaply, but can contain ambiguous questions or incorrect answers. Duplicate questions must not be counted twice. Short source-grounded references avoid penalizing the model for omitting claims that the source never supported.

An LLM judge provides semantic grading, but its grades are not objective accuracy: using the same model to generate and judge answers can introduce correlated errors. Faithfulness, answer relevancy, context precision, and context recall measure different failures. An answer can be faithful yet incomplete, or relevant but unsupported. The deliberately false answer and irrelevant context are negative controls whose measured scores should be compared with the originals.

Live results remain pending; I cannot claim that either dataset scored better. After execution, compare per-question failures, manually review disagreements, and try a different chunk size or retrieval `k` on the same held-out questions. Keep model, questions, and references fixed so comparisons are meaningful.
